# 움직임 시험 1 — 블로킹(움직임만 잡은 밑그림) → LoRA 그림→그림(img2img)으로 장마다 다시 그리기 (Kaggle·Colab T4, 학습 없음)
목적: 움직임이 정해진 밑그림을 컬러 LoRA가 나카무라 화풍으로 장마다 다시 그릴 수 있는지, 장 사이 깜빡임이 얼마나인지 본다.
- 입력 A `num`: 숫자로 만든 블로킹 15장 — 나카무라 폭발 실측 타이밍(tools/fx_blocking.py). 입력 B `clip`: 나카무라 c0348 실루엣·움직임만 남긴 블로킹 12장(tools/fx_blocking_clip.py).
- LoRA: 컬러 3차 3바퀴. 부정어는 기본. 강도(원래 그림을 얼마나 바꿀지) 0.4 · 0.55 · 0.7, 모든 장 같은 시드(1234)·같은 글.
- 2026-10-26 고침: sd-scripts 환경이 Kaggle·Colab 파이썬 3.13에서 깨져(첫 실행 6분 만에 그림 없이 끝남) diffusers로 바꿈(시험 2와 같은 방식).
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 약 40분. Colab: 런타임 T4 GPU → 모두 실행(마지막 칸이 zip을 내려받게 함).
- 끝나면 `motion1_result.zip`(나란히 영상 `motion1.mp4` + 장별 그림 + 깜빡임 수치)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·블로킹 두 가지(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0"
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
import os, glob
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL 그림→그림 + LoRA 합치기
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
pipe = StableDiffusionXLImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
pipe.fuse_lora(lora_scale=1.0); pipe.unload_lora_weights()
pipe.to("cuda")
pipe.vae.to(torch.float32)  # VAE는 fp32(검은 그림·NaN 방지, 시험 1의 --no_half_vae와 같음)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) 강도별 그리기(장마다 따로). 같은 블로킹(걸이 장)은 같은 결과를 다시 씀
import numpy as np, json
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
STEPS, CFG, SEED = 28, 6.0, 1234
def run(init, s):
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=init, strength=s, num_inference_steps=STEPS, guidance_scale=CFG,
                    generator=g, width=1344, height=768).images[0]
def diff(a, b):
    return float(np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean())
log = {}
for name in ("num", "clip"):
    blk = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"/tmp/blk_{name}/*.png"))]
    d0 = f"{OUT}/m/{name}/blocking"; os.makedirs(d0, exist_ok=True)
    for i, b in enumerate(blk):
        b.save(f"{d0}/f{i:03d}.png")
    for s in (0.4, 0.55, 0.7):
        cache, ims = {}, []
        for b in blk:
            key = np.asarray(b).tobytes()
            if key not in cache:
                cache[key] = run(b, s)
            ims.append(cache[key])
        d = f"{OUT}/m/{name}/s{s}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(ims):
            im.save(f"{d}/f{i:03d}.png")
        fl = np.mean([abs(diff(ims[i + 1], ims[i]) - diff(blk[i + 1], blk[i])) for i in range(len(ims) - 1)])
        log[f"{name} s{s}"] = round(float(fl), 2)
        print(name, s, len(ims), "장, 깜빡임", log[f"{name} s{s}"])
json.dump(log, open(f"{OUT}/motion1_flicker.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | 0.4 | 0.55 | 0.7. 24fps, 3번 반복
import cv2
rows = {n: {k: sorted(glob.glob(f"{OUT}/m/{n}/{k}/*.png")) for k in ("blocking", "s0.4", "s0.55", "s0.7")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion1.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1344, 384))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion1_result.zip m motion1.mp4 motion1_flicker.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion1_result.zip")
